# Tiny LLM: training notebook (Kaggle & Colab)

Train and fine-tune **Tiny LLM** on a **free Kaggle or Colab GPU**.
This notebook is **self-contained**: the whole `tiny_llm` package and its scripts are written to disk
by the cells below, identical to the [tiny-llm](https://github.com/benisalla/tiny-llm) repo.
Just upload it and press **Run all**.

1. Setup
2. The code (`tiny_llm` package + scripts)
3. Sanity check on simple data: addition from scratch, LoRA reading comprehension
4. Pre-training on OpenWebText (the Kaggle dataset `benallaismail/gpt-data`)
5. Chat with another small LLM loaded into our architecture
6. Download the checkpoints

**Kaggle:** Settings → Accelerator → GPU, and Internet → on.
**Colab:** Runtime → Change runtime type → GPU.

## 1. Setup

In [ ]:
import os, sys, subprocess

ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else os.getcwd())
os.chdir(WORK)
print("working directory:", WORK)

# torch is preinstalled on Kaggle and Colab
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.36", "tiktoken"], check=True)
os.makedirs("tiny_llm", exist_ok=True)
os.makedirs("scripts", exist_ok=True)
os.environ["PYTHONPATH"] = WORK  # so `python scripts/...` finds the package

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. The code

The cells below write the package (`tiny_llm/`) and the command-line scripts (`scripts/`).
You don't need to read them to use the notebook; they are here so it runs without GitHub.

In [ ]:
%%writefile tiny_llm/__init__.py
"""Tiny LLM: a small LLM built from scratch that can load GPT-2, Llama, Qwen2 and Pythia weights."""
from tiny_llm.checkpoint import load_checkpoint, save_checkpoint
from tiny_llm.config import ModelConfig, TrainConfig
from tiny_llm.families import FAMILIES
from tiny_llm.generation import answer_me, generate_text, stream_answer, stream_chat, stream_tokens
from tiny_llm.lora import LoraEmbedding, LoraLinear, mark_only_lora_as_trainable, merge_lora
from tiny_llm.model import TinyLLM
from tiny_llm.tokenizer import create_tokenizer, format_qa_prompt, format_qa_sample

__version__ = "2.0.0"

In [ ]:
%%writefile tiny_llm/attention.py
import torch
from torch import nn
from torch.nn import functional as F

from tiny_llm.lora import LoraLinear
from tiny_llm.rope import RotaryEmbedding


class KVCache:
    """
        Keys and values of the tokens already processed by one attention layer.
        During generation each new token only computes its own q, k, v and attends to the cached ones,
        instead of recomputing the whole sequence at every step.
    """

    def __init__(self):
        self.k = self.v = None

    @property
    def length(self):
        return 0 if self.k is None else self.k.size(2)

    def update(self, k, v):
        self.k = k if self.k is None else torch.cat((self.k, k), dim=2)
        self.v = v if self.v is None else torch.cat((self.v, v), dim=2)
        return self.k, self.v


class CausalSelfAttention(nn.Module):
    """
        Causal Self-Attention Module
        ( why causal? because each token can only attend to itself and the tokens before it )

        Supports multi-head attention (GPT-2, Pythia) and grouped-query attention (Llama, Qwen2), where
        several query heads share the same key/value head (n_kv_head < n_head: smaller KV cache).

        Tensor          Type            Shape
        ===========================================================================
        input           float           (batch_size, seq_len, n_embd)
        ---------------------------------------------------------------------------
        output          float           (batch_size, seq_len, n_embd)
        ===========================================================================
    """

    def __init__(self, config, rotary=None):
        super().__init__()
        """
            dropout ~= lora_dropout since Pre-Trained W is frozen
            we can either introduce new dropout (lora_dropout) or use dropout
        """
        self.n_head = config.n_head
        self.n_kv_head = config.kv_heads
        self.head_dim = config.head_size
        self.drop_rate = config.drop_rate
        q_dim, kv_dim = self.n_head * self.head_dim, self.n_kv_head * self.head_dim
        self.split_sizes = [q_dim, kv_dim, kv_dim]

        lora_args = dict(rank=config.lora_rank, lora_alpha=config.lora_alpha, lora_dropout=config.lora_dropout)
        # query, key, value projections for all heads, in one batch
        self.c_attn = LoraLinear(config.n_embd, q_dim + 2 * kv_dim, bias=config.use_qkv_bias, **lora_args)
        # output projection
        self.c_proj = LoraLinear(q_dim, config.n_embd, bias=config.bias, **lora_args)
        self.res_drop = nn.Dropout(config.drop_rate)

        # the model shares one rotary table between all layers; standalone layers build their own
        self.rotary = None
        if config.pos_encoding == "rope":
            self.rotary = rotary or RotaryEmbedding(self.head_dim, config.block_size, config.rope_theta,
                                                    config.rotary_pct)

    def forward(self, x, kv_cache=None):
        B, T, _ = x.size()
        start_pos = kv_cache.length if kv_cache is not None else 0

        q, k, v = self.c_attn(x).split(self.split_sizes, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hs)
        k = k.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)  # (B, n_kv, T, hs)
        v = v.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)  # (B, n_kv, T, hs)

        if self.rotary is not None:
            q, k = self.rotary(q, k, start_pos)
        if kv_cache is not None:
            k, v = kv_cache.update(k, v)  # (B, n_kv, start_pos + T, hs)

        if self.n_kv_head != self.n_head:  # grouped-query attention: share each kv head between query heads
            k = k.repeat_interleave(self.n_head // self.n_kv_head, dim=1)
            v = v.repeat_interleave(self.n_head // self.n_kv_head, dim=1)

        S = k.size(2)
        if T == 1:  # one new token: it may look at everything before it
            mask, is_causal = None, False
        elif S == T:  # no cache: plain causal mask
            mask, is_causal = None, True
        else:  # new tokens on top of a cache: token i sees cached tokens + new tokens up to i
            mask, is_causal = torch.ones(T, S, dtype=torch.bool, device=x.device).tril(diagonal=S - T), False

        # flash attention when available: softmax(q k^T / sqrt(hs)) v
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal,
                                           dropout_p=self.drop_rate if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)  # re-assemble all heads

        return self.res_drop(self.c_proj(y))

In [ ]:
%%writefile tiny_llm/block.py
from torch import nn

from tiny_llm.attention import CausalSelfAttention
from tiny_llm.mlp import MLP
from tiny_llm.norm import build_norm


class DBlock(nn.Module):
    """
        Transformer (Decoder) Block: communication layer (att mechanism) + computation layer (feedforward).
        Pre-normalisation, as in GPT-2 and every model after it.

            sequential (default):        x = x + attn(ln_1(x));  x = x + mlp(ln_2(x))
            parallel   (Pythia/NeoX):    x = x + attn(ln_1(x)) + mlp(ln_2(x))

        Tensor          Type            Shape
        ===========================================================================
        input           float           (batch_size, seq_len, n_embd)
        ---------------------------------------------------------------------------
        output          float           (batch_size, seq_len, n_embd)
        ===========================================================================
    """

    def __init__(self, config, rotary=None):
        super().__init__()
        # names match Hugging Face GPT-2 (other families are renamed in tiny_llm/families.py)
        self.ln_1 = build_norm(config)
        self.attn = CausalSelfAttention(config, rotary)
        self.ln_2 = build_norm(config)
        self.mlp = MLP(config)
        self.parallel_residual = config.parallel_residual

    def forward(self, x, kv_cache=None):
        attn_out = self.attn(self.ln_1(x), kv_cache)
        if self.parallel_residual:
            return x + attn_out + self.mlp(self.ln_2(x))
        x = x + attn_out
        return x + self.mlp(self.ln_2(x))

In [ ]:
%%writefile tiny_llm/checkpoint.py
import os

import torch

from tiny_llm.config import ModelConfig
from tiny_llm.model import TinyLLM

# old module names -> current names (the first version of this repo used attn_net / ff_mlp)
LEGACY_RENAMES = {'.attn_net.': '.attn.', '.ff_mlp.': '.mlp.'}
UNWANTED_PREFIX = '_orig_mod.'  # added by torch.compile


def clean_state_dict(state_dict):
    cleaned = {}
    for key, value in state_dict.items():
        if key.startswith(UNWANTED_PREFIX):
            key = key[len(UNWANTED_PREFIX):]
        for old, new in LEGACY_RENAMES.items():
            key = key.replace(old, new)
        cleaned[key] = value
    return cleaned


def save_checkpoint(path, model, optimizer=None, model_args=None, iter_num=0, best_val_loss=float('inf'),
                    extra=None):
    model = getattr(model, '_orig_mod', model)  # unwrap torch.compile
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    checkpoint = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict() if optimizer is not None else None,
        'model_args': model_args or model.config.to_dict(),
        'iter_num': iter_num,
        'best_val_loss': best_val_loss,
        **(extra or {}),
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved to {path} :)")


def load_checkpoint(path, device='cpu'):
    """Return (model, checkpoint dict). The model is built from the config stored in the checkpoint."""
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    state_dict = clean_state_dict(checkpoint['model'])
    model_args = dict(checkpoint['model_args'])

    # old checkpoints had LoRA layers but no `lora_rank` in model_args: infer it from the weights
    lora_keys = [k for k in state_dict if k.endswith('lora_A')]
    if lora_keys and not model_args.get('lora_rank'):
        model_args['lora_rank'] = state_dict[lora_keys[0]].size(0)

    model = TinyLLM(ModelConfig.from_dict(model_args))
    model.load_state_dict(state_dict)
    model.to(device)
    checkpoint['model_args'] = model.config.to_dict()
    return model, checkpoint

In [ ]:
%%writefile tiny_llm/config.py
from dataclasses import asdict, dataclass, fields
from typing import Optional


@dataclass
class ModelConfig:
    """
    Architecture of the model (everything needed to rebuild it from a checkpoint).

    The defaults describe GPT-2. Other small LLMs are the same decoder-only transformer with a few
    design choices changed. Those choices are the fields of the second block below, and
    `tiny_llm/families.py` lists, for each supported family, which of them it changes.
    """

    # ---- size ----
    block_size: int = 1024  # context length
    vocab_size: int = 50304  # 50257 GPT-2 tokens + our special tokens, padded to a multiple of 64
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    drop_rate: float = 0.0  # 0.0 for pre-training, ~0.1 for fine-tuning (prevent it from learning too much)
    bias: bool = True  # biases in Linear layers and LayerNorms

    # ---- what changes between model families (defaults = GPT-2) ----
    n_kv_head: int = 0  # 0 -> same as n_head. Fewer KV heads = grouped-query attention (Llama, Qwen2)
    head_dim: int = 0  # 0 -> n_embd // n_head
    qkv_bias: Optional[bool] = None  # None -> same as `bias`. Qwen2: biases on q/k/v only
    norm_type: str = "layernorm"  # "layernorm" (GPT-2, Pythia) or "rmsnorm" (Llama, Qwen2)
    norm_eps: float = 1e-5
    pos_encoding: str = "learned"  # "learned" absolute positions (GPT-2) or "rope" rotary (Llama, Qwen2, Pythia)
    rope_theta: float = 10000.0  # RoPE base frequency
    rotary_pct: float = 1.0  # fraction of each head that is rotated (Pythia: 0.25)
    activation: str = "gelu_tanh"  # "gelu_tanh" (GPT-2), "gelu" (Pythia), "silu" (Llama, Qwen2), "relu"
    gated_mlp: bool = False  # SwiGLU MLP: down(act(gate(x)) * up(x)) (Llama, Qwen2)
    intermediate_size: int = 0  # MLP hidden size, 0 -> 4 * n_embd
    parallel_residual: bool = False  # x + attn(ln_1(x)) + mlp(ln_2(x)) instead of sequential (Pythia)
    tie_embeddings: bool = True  # lm_head shares the token embedding matrix (GPT-2, SmolLM2, Qwen2.5-0.5B)

    # for Mrs. LoRA (lora_rank = 0 means LoRA is disabled)
    lora_rank: int = 0
    lora_alpha: float = 16.0
    lora_dropout: float = 0.05

    def __post_init__(self):
        assert self.head_dim or self.n_embd % self.n_head == 0, "n_embd % n_head should be 0."
        assert self.n_head % self.kv_heads == 0, "n_head must be a multiple of n_kv_head"
        assert self.norm_type in ("layernorm", "rmsnorm"), self.norm_type
        assert self.pos_encoding in ("learned", "rope"), self.pos_encoding
        assert self.activation in ("gelu_tanh", "gelu", "silu", "relu"), self.activation

    # derived sizes
    @property
    def kv_heads(self):
        return self.n_kv_head or self.n_head

    @property
    def head_size(self):
        return self.head_dim or self.n_embd // self.n_head

    @property
    def hidden_size(self):
        return self.intermediate_size or 4 * self.n_embd

    @property
    def use_qkv_bias(self):
        return self.bias if self.qkv_bias is None else self.qkv_bias

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_dict(cls, d):
        """Build a config from a dict, ignoring unknown keys (e.g. old checkpoints)."""
        names = {f.name for f in fields(cls)}
        return cls(**{k: v for k, v in d.items() if k in names})


@dataclass
class TrainConfig:
    """Everything about the optimisation loop (pre-training and fine-tuning)."""

    # "scratch", "resume", or a Hugging Face model id / local folder of a supported family
    # ("gpt2", "distilgpt2", "HuggingFaceTB/SmolLM2-135M", "Qwen/Qwen2.5-0.5B", "EleutherAI/pythia-70m", ...)
    init_from: str = "scratch"
    tokenizer: str = ""  # Hugging Face tokenizer, "" -> the pre-trained model's own (or "gpt2" from scratch)

    # data
    data_dir: str = "./data"
    batch_size: int = 12  # sequences per micro-batch (pre-training)
    toks_per_batch: int = 6000  # tokens per micro-batch (fine-tuning, batches are built by length)
    grad_accum_steps: int = 40  # micro-batches per optimizer step

    # optimizer
    learning_rate: float = 6e-4  # change this to 3e-5 in fine-tuning stage
    weight_decay: float = 1e-1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0

    # lr schedule (linear warmup + cosine decay)
    decay_lr: bool = True
    warmup_iters: int = 2000
    lr_decay_iters: int = 600000
    min_lr: float = 6e-5
    max_iters: int = 600000

    # evaluation / logging / checkpointing
    eval_interval: int = 250
    eval_iters: int = 200
    log_interval: int = 10
    always_save_checkpoint: bool = False  # if False, only save when val loss improves
    out_dir: str = "./checkpoints"
    ckpt_name: str = "ckpt.pt"  # checkpoints are written to out_dir/ckpt_name
    resume_from: str = ""  # checkpoint to load with init_from="resume" (default: out_dir/ckpt_name)

    # fine-tuning
    loss_on_answer_only: bool = True  # only learn to predict the answer, not the context/question
    train_embeddings: bool = True  # with LoRA: keep the (tied) token embeddings trainable so new special tokens get learned
    max_seq_len: int = 400  # drop fine-tuning samples longer than this (in tokens)

    # system
    device: str = "auto"  # "auto", "cpu", "cuda", "mps"
    dtype: str = "auto"  # "auto", "float32", "bfloat16", "float16"
    compile: bool = False
    seed: int = 2000

In [ ]:
%%writefile tiny_llm/data.py
import json
import os
from random import Random

import numpy as np
import torch
from torch.nn.utils.rnn import pad_sequence

from tiny_llm.tokenizer import ANS, format_qa_sample

IGNORE_INDEX = -1  # targets with this value don't contribute to the loss


def load_bin(data_dir, split):
    """Token ids stored as a flat uint16 binary file (see scripts/prepare_data.py)."""
    return np.memmap(os.path.join(data_dir, f"{split}.bin"), dtype=np.uint16, mode='r')


def get_batch(data, batch_size, block_size, device):
    """Sample `batch_size` random windows from a flat token array. y is x shifted by one."""
    assert len(data) > block_size, f"dataset has {len(data)} tokens, need more than block_size={block_size}"
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i + block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i + 1:i + 1 + block_size]).astype(np.int64)) for i in ix])
    if torch.device(device).type == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


def _field(sample, name):
    # the original dataset stores its keys reversed ("txetnoc", "noitseuq", "rewsna"), accept both
    return sample.get(name, sample.get(name[::-1], ""))


def load_qa_samples(path):
    """JSON list of {"context": ..., "question": ..., "answer": ...} -> list of formatted strings."""
    with open(path, 'r', encoding='utf-8') as json_file:
        raw_data = json.load(json_file)
    return [format_qa_sample(_field(s, "question"), _field(s, "answer"), _field(s, "context")) for s in raw_data]


class BatchLoader:
    """
    Infinite iterator of padded (x, y) batches for question/answer fine-tuning.

    Samples are sorted by length and grouped so that each batch holds about `toks_in_batch` tokens
    (little padding waste). Training batches are shuffled every epoch.

    Args:
        data_folder (str): folder containing `{split}.json`.
        split (str): "train", "val" or "test".
        tokenizer: tokenizer with our special tokens (see tiny_llm.tokenizer).
        toks_in_batch (int): token budget per batch.
        max_len (int): samples longer than this (in tokens) are dropped.
        answer_only (bool): if True, targets before `<ans>` are ignored (learn to answer, not to ask).
    """

    def __init__(self, data_folder, split, tokenizer, toks_in_batch, device="cpu", max_len=400,
                 answer_only=True, seed=0):
        self.toks_in_batch = toks_in_batch
        self.training = (split == "train")
        self.tokenizer = tokenizer
        self.device = device
        self.answer_only = answer_only
        self.rng = Random(seed)
        self.ans_id = tokenizer.convert_tokens_to_ids(ANS)

        texts = load_qa_samples(os.path.join(data_folder, f"{split}.json"))
        encoded = [tokenizer.encode(t) for t in texts]
        # filter our data and pre-sort by lengths
        self.data = sorted((ids for ids in encoded if 2 < len(ids) <= max_len), key=len)
        assert self.data, f"no usable samples in {split}.json"
        self.create_batches()

    def create_batches(self):
        self.batches = []
        batch = []
        for ids in self.data:  # data is sorted, so the last sample of a batch is the longest
            if batch and len(ids) * (len(batch) + 1) > self.toks_in_batch:
                self.batches.append(batch)
                batch = []
            batch.append(ids)
        if batch:
            self.batches.append(batch)
        if self.training:
            self.rng.shuffle(self.batches)
        self.n_batches = len(self.batches)
        self.current = -1

    def __iter__(self):
        return self

    def __len__(self):
        return len(self.batches)

    def __next__(self):
        self.current += 1
        if self.current >= self.n_batches:  # new epoch
            self.create_batches()
            self.current = 0
        seqs = [torch.LongTensor(ids) for ids in self.batches[self.current]]

        input_ids = pad_sequence(seqs, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        targets = pad_sequence(seqs, batch_first=True, padding_value=IGNORE_INDEX)
        if self.answer_only:
            for row, ids in enumerate(seqs):
                ans_pos = (ids == self.ans_id).nonzero()
                if len(ans_pos):
                    targets[row, :ans_pos[0, 0] + 1] = IGNORE_INDEX  # keep only tokens after <ans>

        x_batch = input_ids[:, :-1].to(self.device)
        y_batch = targets[:, 1:].to(self.device)
        return x_batch, y_batch

In [ ]:
%%writefile tiny_llm/families.py
"""
Model families: what changes for each type of small LLM.

Every model below is the same decoder-only transformer as GPT-2 (embeddings -> N x [norm, attention,
norm, MLP] -> norm -> lm_head). They only differ by a handful of design choices, which are the fields
of `ModelConfig`. Each family here says:

    1. `changes`    : which choices differ from GPT-2 (human readable, also printed in the README table)
    2. `to_config`  : how to read them from a Hugging Face config
    3. `convert`    : how its weight names map onto ours (our names follow GPT-2)

To support a new family: write these three things and call `register(...)`. If it needs a design choice
that doesn't exist yet (e.g. Gemma scales the embeddings by sqrt(n_embd)), add a field to ModelConfig and
use it in the layer.
"""
from dataclasses import dataclass
from typing import Callable, Dict, Tuple

import torch


@dataclass(frozen=True)
class Family:
    model_type: str  # Hugging Face `config.model_type`
    examples: Tuple[str, ...]  # small checkpoints on the Hugging Face Hub
    changes: Dict[str, str]  # ModelConfig field -> what this family uses (vs GPT-2)
    to_config: Callable  # hf_config -> dict of ModelConfig arguments
    convert: Callable  # (hf_state_dict, ModelConfig) -> our state_dict


FAMILIES: Dict[str, Family] = {}


def register(family):
    FAMILIES[family.model_type] = family
    return family


def get_family(model_type):
    if model_type not in FAMILIES:
        raise NotImplementedError(f"model type {model_type!r} is not supported yet. "
                                  f"Supported: {sorted(FAMILIES)}. See tiny_llm/families.py to add one.")
    return FAMILIES[model_type]


# --------------------------------------------------------------------------------------------- helpers

class _Keys:
    """Pops Hugging Face weights by name so we can check at the end that every weight was used."""

    IGNORED = ('.attn.bias', '.attn.masked_bias', 'rotary_emb.inv_freq', '.attention.bias', '.attention.masked_bias')

    def __init__(self, state_dict):
        self.sd = dict(state_dict)

    def pop(self, key):
        return self.sd.pop(key)

    def pop_optional(self, key):
        return self.sd.pop(key, None)

    def check_all_used(self):
        left = [k for k in self.sd if not k.endswith(self.IGNORED)]
        if left:
            raise ValueError(f"these weights were not converted (unsupported variant?): {left[:10]}")


def _rope(c):
    """RoPE settings, for both transformers v4 (rope_theta, rope_scaling) and v5 (rope_parameters)."""
    params = dict(getattr(c, "rope_parameters", None) or getattr(c, "rope_scaling", None) or {})
    theta = params.get("rope_theta", getattr(c, "rope_theta", getattr(c, "rotary_emb_base", 10000.0)))
    pct = params.get("partial_rotary_factor", getattr(c, "partial_rotary_factor", getattr(c, "rotary_pct", 1.0)))
    rope_type = params.get("rope_type", params.get("type", "default"))
    if rope_type != "default":
        raise NotImplementedError(f"RoPE scaling {rope_type!r} is not implemented (only plain RoPE)")
    return float(theta), float(pct)


def _head_dim(c, n_embd, n_head):
    head_dim = getattr(c, "head_dim", None) or n_embd // n_head
    return 0 if head_dim == n_embd // n_head else head_dim


# --------------------------------------------------------------------------------------------- GPT-2

def _gpt2_config(c):
    act = {"gelu_new": "gelu_tanh", "gelu_pytorch_tanh": "gelu_tanh", "gelu": "gelu", "relu": "relu"}
    return dict(n_layer=c.n_layer, n_head=c.n_head, n_embd=c.n_embd, block_size=c.n_positions,
                vocab_size=c.vocab_size, bias=True, norm_eps=c.layer_norm_epsilon,
                activation=act[c.activation_function], intermediate_size=c.n_inner or 0)


def _gpt2_convert(sd_hf, cfg):
    keys = _Keys(sd_hf)
    sd = {}
    # HF GPT-2 uses Conv1D, i.e. weights are stored transposed compared to nn.Linear
    transposed = ('attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight')
    for k in list(keys.sd):
        if k.endswith(_Keys.IGNORED):
            continue
        name = k if k.startswith(('transformer.', 'lm_head.')) else f"transformer.{k}"  # some saves drop the prefix
        value = keys.pop(k)
        sd[name] = value.t() if k.endswith(transposed) else value
    sd.setdefault("lm_head.weight", sd["transformer.wte.weight"])  # tied, some saves only keep wte
    keys.check_all_used()
    return sd


register(Family(
    model_type="gpt2",
    examples=("gpt2", "distilgpt2", "gpt2-medium", "gpt2-large", "gpt2-xl"),
    changes={},  # the reference
    to_config=_gpt2_config,
    convert=_gpt2_convert,
))


# --------------------------------------------------------------------------------------------- Llama / Qwen2

def _llama_config(c, qkv_bias=False):
    if getattr(c, "mlp_bias", False) or getattr(c, "attention_bias", False):
        raise NotImplementedError("Llama variants with attention_bias / mlp_bias are not supported")
    theta, pct = _rope(c)
    return dict(n_layer=c.num_hidden_layers, n_head=c.num_attention_heads, n_embd=c.hidden_size,
                n_kv_head=c.num_key_value_heads, head_dim=_head_dim(c, c.hidden_size, c.num_attention_heads),
                block_size=c.max_position_embeddings, vocab_size=c.vocab_size,
                bias=False, qkv_bias=qkv_bias, norm_type="rmsnorm", norm_eps=c.rms_norm_eps,
                pos_encoding="rope", rope_theta=theta, rotary_pct=pct,
                activation=c.hidden_act, gated_mlp=True, intermediate_size=c.intermediate_size,
                tie_embeddings=bool(c.tie_word_embeddings))


def _qwen2_config(c):
    if getattr(c, "use_sliding_window", False):
        raise NotImplementedError("Qwen2 sliding-window attention is not supported")
    return _llama_config(c, qkv_bias=True)


def _llama_convert(sd_hf, cfg):
    keys = _Keys(sd_hf)
    sd = {"transformer.wte.weight": keys.pop("model.embed_tokens.weight"),
          "transformer.ln_f.weight": keys.pop("model.norm.weight")}
    for i in range(cfg.n_layer):
        hf, ours = f"model.layers.{i}.", f"transformer.h.{i}."
        sd[ours + "ln_1.weight"] = keys.pop(hf + "input_layernorm.weight")
        sd[ours + "ln_2.weight"] = keys.pop(hf + "post_attention_layernorm.weight")
        # separate q, k, v projections -> one fused c_attn
        sd[ours + "attn.c_attn.weight"] = torch.cat([keys.pop(hf + f"self_attn.{p}_proj.weight") for p in "qkv"])
        if cfg.use_qkv_bias:
            sd[ours + "attn.c_attn.bias"] = torch.cat([keys.pop(hf + f"self_attn.{p}_proj.bias") for p in "qkv"])
        sd[ours + "attn.c_proj.weight"] = keys.pop(hf + "self_attn.o_proj.weight")
        sd[ours + "mlp.c_gate.weight"] = keys.pop(hf + "mlp.gate_proj.weight")
        sd[ours + "mlp.c_fc.weight"] = keys.pop(hf + "mlp.up_proj.weight")
        sd[ours + "mlp.c_proj.weight"] = keys.pop(hf + "mlp.down_proj.weight")
    lm_head = keys.pop_optional("lm_head.weight")
    sd["lm_head.weight"] = sd["transformer.wte.weight"] if cfg.tie_embeddings or lm_head is None else lm_head
    keys.check_all_used()
    return sd


LLAMA_CHANGES = {
    "norm_type": "rmsnorm (no bias)",
    "pos_encoding": "rope (no position embedding matrix)",
    "activation / gated_mlp": "silu, gated (SwiGLU)",
    "n_kv_head": "grouped-query attention",
    "bias": "False everywhere",
    "tie_embeddings": "depends on the checkpoint",
}

register(Family(
    model_type="llama",
    examples=("HuggingFaceTB/SmolLM2-135M", "HuggingFaceTB/SmolLM2-360M", "HuggingFaceTB/SmolLM2-135M-Instruct",
              "TinyLlama/TinyLlama-1.1B-Chat-v1.0"),
    changes=LLAMA_CHANGES,
    to_config=_llama_config,
    convert=_llama_convert,
))

register(Family(
    model_type="qwen2",
    examples=("Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-0.5B-Instruct"),
    changes={**LLAMA_CHANGES, "bias": "False, except q/k/v projections (qkv_bias=True)"},
    to_config=_qwen2_config,
    convert=_llama_convert,
))


# --------------------------------------------------------------------------------------------- GPT-NeoX (Pythia)

def _neox_config(c):
    theta, pct = _rope(c)
    return dict(n_layer=c.num_hidden_layers, n_head=c.num_attention_heads, n_embd=c.hidden_size,
                block_size=c.max_position_embeddings, vocab_size=c.vocab_size, bias=True,
                norm_eps=c.layer_norm_eps, pos_encoding="rope", rope_theta=theta, rotary_pct=pct,
                activation={"gelu": "gelu", "gelu_new": "gelu_tanh", "relu": "relu"}[c.hidden_act],
                intermediate_size=c.intermediate_size, parallel_residual=bool(c.use_parallel_residual),
                tie_embeddings=bool(c.tie_word_embeddings))


def _neox_convert(sd_hf, cfg):
    keys = _Keys(sd_hf)
    nh, hd = cfg.n_head, cfg.head_size

    def unfuse(t):
        # NeoX packs qkv per head: [h0q h0k h0v h1q ...] -> ours: [all q | all k | all v]
        t = t.view(nh, 3, hd, *t.shape[1:])
        return torch.cat([t[:, j].reshape(nh * hd, *t.shape[3:]) for j in range(3)])

    sd = {"transformer.wte.weight": keys.pop("gpt_neox.embed_in.weight")}
    for p in ("weight", "bias"):
        sd[f"transformer.ln_f.{p}"] = keys.pop(f"gpt_neox.final_layer_norm.{p}")
    for i in range(cfg.n_layer):
        hf, ours = f"gpt_neox.layers.{i}.", f"transformer.h.{i}."
        for p in ("weight", "bias"):
            sd[ours + f"ln_1.{p}"] = keys.pop(hf + f"input_layernorm.{p}")
            sd[ours + f"ln_2.{p}"] = keys.pop(hf + f"post_attention_layernorm.{p}")
            sd[ours + f"attn.c_attn.{p}"] = unfuse(keys.pop(hf + f"attention.query_key_value.{p}"))
            sd[ours + f"attn.c_proj.{p}"] = keys.pop(hf + f"attention.dense.{p}")
            sd[ours + f"mlp.c_fc.{p}"] = keys.pop(hf + f"mlp.dense_h_to_4h.{p}")
            sd[ours + f"mlp.c_proj.{p}"] = keys.pop(hf + f"mlp.dense_4h_to_h.{p}")
    lm_head = keys.pop_optional("embed_out.weight")  # transformers v4 name
    lm_head = keys.pop("lm_head.weight") if lm_head is None else lm_head  # transformers v5 name
    sd["lm_head.weight"] = sd["transformer.wte.weight"] if cfg.tie_embeddings else lm_head
    keys.check_all_used()
    return sd


register(Family(
    model_type="gpt_neox",
    examples=("EleutherAI/pythia-70m", "EleutherAI/pythia-160m", "EleutherAI/pythia-410m"),
    changes={
        "pos_encoding / rotary_pct": "rope on 25% of each head",
        "parallel_residual": "True: attention and MLP read the same input",
        "activation": "gelu (exact)",
        "tie_embeddings": "False (separate output matrix)",
    },
    to_config=_neox_config,
    convert=_neox_convert,
))


# --------------------------------------------------------------------------------------------- summary

def markdown_table():
    """What changes for each family, as a markdown table (scripts/list_families.py, README)."""
    rows = ["| family (`model_type`) | what changes vs GPT-2 | small checkpoints |", "| --- | --- | --- |"]
    for f in FAMILIES.values():
        changes = "<br>".join(f"`{k}`: {v}" for k, v in f.changes.items()) or "reference architecture"
        rows.append(f"| `{f.model_type}` | {changes} | {', '.join(f'`{e}`' for e in f.examples)} |")
    return "\n".join(rows)

In [ ]:
%%writefile tiny_llm/generation.py
"""High level inference helpers: text completion, question answering and chat, with streaming."""
import torch

from tiny_llm.tokenizer import PAD, format_qa_prompt, stop_token_ids


def _banned_ids(tokenizer, model):
    """Padding and the unused rows added to round the vocabulary up must never be sampled."""
    banned = [tokenizer.convert_tokens_to_ids(PAD)]
    banned += list(range(len(tokenizer), model.config.vocab_size))
    return banned


def stream_tokens(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_k=50, top_p=0.95,
                  device=None, stop_at_eos=True, add_special_tokens=True):
    """
    Yield the generated text piece by piece (decoded incrementally so multi-byte characters
    never show up half written). The prompt itself is not yielded.
    """
    device = device or next(model.parameters()).device
    model.eval()
    ids = tokenizer.encode(prompt, add_special_tokens=add_special_tokens)
    # keep room for the answer inside the context window
    max_prompt = max(1, model.config.block_size - max_new_tokens)
    ids = ids[-max_prompt:]
    x = torch.tensor(ids, dtype=torch.long, device=device)[None, ...]

    stop_ids = stop_token_ids(tokenizer)
    generated, emitted = [], ""
    for tok in model.stream_generate(x, max_new_tokens, temperature=temperature, top_k=top_k, top_p=top_p,
                                     eos_token_id=stop_ids if stop_at_eos else None,
                                     banned_token_ids=_banned_ids(tokenizer, model)):
        tok = tok.item()
        if stop_at_eos and tok in stop_ids:
            break
        generated.append(tok)
        text = tokenizer.decode(generated, skip_special_tokens=True)
        if text.endswith("�"):  # incomplete utf-8 character, wait for the next token
            continue
        if len(text) > len(emitted):
            yield text[len(emitted):]
            emitted = text


def generate_text(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0, top_k=100, top_p=None,
                  device=None):
    """Complete `prompt` and return prompt + completion."""
    return prompt + "".join(stream_tokens(model, tokenizer, prompt, max_new_tokens, temperature, top_k, top_p,
                                          device))


def stream_answer(model, tokenizer, question, context="", **kwargs):
    """Question answering with a fine-tuned model (format: <sost>context<ques>question<ans>answer<eost>)."""
    return stream_tokens(model, tokenizer, format_qa_prompt(question, context), **kwargs)


def answer_me(model, tokenizer, question, context="", **kwargs):
    return "".join(stream_answer(model, tokenizer, question, context, **kwargs)).strip()


def has_chat_template(tokenizer):
    return bool(getattr(tokenizer, "chat_template", None))


def stream_chat(model, tokenizer, messages, **kwargs):
    """
    Chat with an instruct model (SmolLM2-Instruct, Qwen2.5-Instruct, TinyLlama-Chat...) using its own
    chat template. messages = [{"role": "system"|"user"|"assistant", "content": "..."}, ...]
    """
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # the template already contains the special tokens (e.g. <s>), don't add them twice
    return stream_tokens(model, tokenizer, prompt, add_special_tokens=False, **kwargs)

In [ ]:
%%writefile tiny_llm/lora.py
"""
LoRA: Low-Rank Adaptation of Large Language Models (https://arxiv.org/abs/2106.09685)

    Y = W*X + (B*A)*X * scale        W is frozen, only A and B (rank r <<< min(in_dim, out_dim)) are trained

With rank = 0 these layers behave exactly like nn.Linear / nn.Embedding, so the same model
is used for pre-training (no LoRA) and for fine-tuning HUGE LLMs (LoRA).
Freezing is NOT done inside the layers: call `mark_only_lora_as_trainable(model)`.
"""
import math

import torch.nn as nn
from torch.nn import functional as F


class LoraLinear(nn.Linear):
    """ Warning: 0 < rank <<< min(in_dim, out_dim), rank = 0 disables LoRA. """

    def __init__(self, in_dim, out_dim, rank=0, lora_alpha=1.0, lora_dropout=0.0, **kwargs):
        super().__init__(in_dim, out_dim, **kwargs)
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.lora_merged = False

        # dropout OR identity f(x) = x
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()

        if rank > 0:  # Y = Llm_w*X + lora_w*X = W*X + B*A*X
            self.lora_A = nn.Parameter(self.weight.new_zeros((rank, in_dim)))
            self.lora_B = nn.Parameter(self.weight.new_zeros((out_dim, rank)))
            self.scale = self.lora_alpha / self.rank  # kind of a hyperparameter
            self.reset_lora_parameters()

    def reset_lora_parameters(self):
        # A random, B = 0  =>  B*A = 0 : at step 0 the model is exactly the pre-trained one
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def merge(self):
        """Fold B*A into W (no extra cost at inference time)."""
        if self.rank > 0 and not self.lora_merged:
            self.weight.data += (self.lora_B @ self.lora_A) * self.scale
            self.lora_merged = True

    def forward(self, inputs):
        llm_w = super().forward(inputs)  # frozen pre-trained weights

        if self.rank <= 0 or self.lora_merged:
            return llm_w  # merged means B*A is already inside W

        lora_w = (self.lora_dropout(inputs) @ self.lora_A.transpose(0, 1) @ self.lora_B.transpose(0, 1)) * self.scale
        return llm_w + lora_w


class LoraEmbedding(nn.Embedding):
    """ Warning: 0 < rank <<< min(num_embeddings, embedding_dim), rank = 0 disables LoRA. """

    def __init__(self, num_embeddings, embedding_dim, rank=0, lora_alpha=1.0, **kwargs):
        super().__init__(num_embeddings, embedding_dim, **kwargs)
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.lora_merged = False

        if rank > 0:
            # Y = Llm_w[x] + (B*A)^T[x]
            self.lora_A = nn.Parameter(self.weight.new_zeros((rank, num_embeddings)))
            self.lora_B = nn.Parameter(self.weight.new_zeros((embedding_dim, rank)))
            self.scale = self.lora_alpha / self.rank
            self.reset_lora_parameters()

    def reset_lora_parameters(self):
        # for embeddings A = 0 and B random (as in microsoft/LoRA), B*A = 0 either way
        nn.init.zeros_(self.lora_A)
        nn.init.normal_(self.lora_B)

    def merge(self):
        if self.rank > 0 and not self.lora_merged:
            self.weight.data += (self.lora_B @ self.lora_A).transpose(0, 1) * self.scale
            self.lora_merged = True

    def forward(self, inputs):
        llm_w = super().forward(inputs)  # old embeds (frozen)

        if self.rank <= 0 or self.lora_merged:
            return llm_w

        lora_emb_A = F.embedding(inputs,
                                 self.lora_A.transpose(0, 1),
                                 self.padding_idx,
                                 self.max_norm,
                                 self.norm_type,
                                 self.scale_grad_by_freq,
                                 self.sparse)  # lora_A act as a lookup table

        lora_w = (lora_emb_A @ self.lora_B.transpose(0, 1)) * self.scale
        return llm_w + lora_w


def mark_only_lora_as_trainable(model, extra_trainable=()):
    """Freeze everything except LoRA params (and params whose name contains one of `extra_trainable`)."""
    for name, param in model.named_parameters():
        param.requires_grad = "lora_" in name or any(key in name for key in extra_trainable)


def merge_lora(model):
    """Merge every LoRA layer into its frozen weights (call before inference)."""
    for module in model.modules():
        if isinstance(module, (LoraLinear, LoraEmbedding)):
            module.merge()
    return model

In [ ]:
%%writefile tiny_llm/mlp.py
from torch import nn

from tiny_llm.lora import LoraLinear

ACTIVATIONS = {
    "gelu_tanh": lambda: nn.GELU(approximate="tanh"),  # GPT-2 ("gelu_new")
    "gelu": nn.GELU,  # Pythia
    "silu": nn.SiLU,  # Llama, Qwen2 (inside SwiGLU)
    "relu": nn.ReLU,
}


class MLP(nn.Module):
    """
        Multi-Layer Perceptron (MLP): computation layer.
        Non-linearity = complex mappings + feature extraction  + ... + processing data (reasoning)

            classic (GPT-2, Pythia):   c_proj( act(c_fc(x)) )
            gated   (Llama, Qwen2):    c_proj( act(c_gate(x)) * c_fc(x) )      a.k.a. SwiGLU with act = SiLU

        Tensor          Type            Shape
        ===========================================================================
        input           float           (batch_size, seq_len, n_embd)
        ---------------------------------------------------------------------------
        output          float           (batch_size, seq_len, n_embd)
        ===========================================================================

    """

    def __init__(self, config):
        super().__init__()
        lora_args = dict(rank=config.lora_rank, lora_alpha=config.lora_alpha, lora_dropout=config.lora_dropout)
        hidden = config.hidden_size
        self.c_fc = LoraLinear(config.n_embd, hidden, bias=config.bias, **lora_args)  # "up" projection
        self.c_gate = LoraLinear(config.n_embd, hidden, bias=config.bias, **lora_args) if config.gated_mlp else None
        self.act = ACTIVATIONS[config.activation]()
        self.c_proj = LoraLinear(hidden, config.n_embd, bias=config.bias, **lora_args)  # "down" projection
        self.dropout = nn.Dropout(config.drop_rate)

    def forward(self, x):
        if self.c_gate is not None:
            h = self.act(self.c_gate(x)) * self.c_fc(x)
        else:
            h = self.act(self.c_fc(x))
        return self.dropout(self.c_proj(h))

In [ ]:
%%writefile tiny_llm/model.py
import inspect
import math

import torch
import torch.nn as nn
from torch.nn import functional as F

from tiny_llm.attention import KVCache
from tiny_llm.block import DBlock
from tiny_llm.lora import LoraEmbedding
from tiny_llm.norm import build_norm
from tiny_llm.rope import RotaryEmbedding


class TinyLLM(nn.Module):
    """
    Tiny LLM: a decoder-only transformer. With the default config it is GPT-2; the config
    switches (see ModelConfig / tiny_llm/families.py) turn it into Llama, Qwen2, Pythia-style models.

    Args:
        config: ModelConfig (vocab_size, block_size, n_layer, n_head, n_embd, ...)

    Attributes:
        transformer: ModuleDict containing the components of the GPT model (Embeddings, Dropout, Blocks, Norm).
        lm_head: Linear layer for language modeling prediction (tied with the token embeddings by default).
        config: Model configuration.
    """

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        rotary = None
        if config.pos_encoding == "rope":  # one cos/sin table shared by every layer
            rotary = RotaryEmbedding(config.head_size, config.block_size, config.rope_theta, config.rotary_pct)
        layers = dict(
            # Note: no LoRA on embeddings. wte is usually tied with lm_head, so a LoRA delta on wte
            # would not be seen by lm_head and merging it would change the model.
            wte=LoraEmbedding(config.vocab_size, config.n_embd),  # token embeddings
            drop=nn.Dropout(config.drop_rate),
            h=nn.ModuleList([DBlock(config, rotary) for _ in range(config.n_layer)]),  # Block X N
            ln_f=build_norm(config),
        )
        if config.pos_encoding == "learned":
            layers["wpe"] = LoraEmbedding(config.block_size, config.n_embd)  # positional embeddings
        self.transformer = nn.ModuleDict(layers)

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        if config.tie_embeddings:
            self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)
        # scaled init of residual projections (GPT-2 paper)
        for param_name, param in self.named_parameters():
            if param_name.endswith('c_proj.weight'):
                torch.nn.init.normal_(param, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
        # re-initialise LoRA params (the generic init above only touched `weight`, but be explicit)
        for module in self.modules():
            if getattr(module, "rank", 0) > 0:
                module.reset_lora_parameters()

    def get_num_params(self, non_embedding=True):
        """Number of params. Position embeddings are excluded by default; token embeddings are
        counted because they are (usually) shared with lm_head."""
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding and "wpe" in self.transformer:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def materialize_(self, device="cpu"):
        """
        After building the model on the meta device and assigning checkpoint weights, create what the
        checkpoint doesn't contain: LoRA matrices and rotary tables. Also restores weight tying.
        """
        for module in self.modules():
            if getattr(module, "rank", 0) > 0 and module.lora_A.is_meta:
                module.lora_A = nn.Parameter(torch.empty_like(module.lora_A, device=device))
                module.lora_B = nn.Parameter(torch.empty_like(module.lora_B, device=device))
                module.reset_lora_parameters()
            if isinstance(module, RotaryEmbedding) and module.inv_freq.is_meta:
                module.reset_buffers(device)
        if self.config.tie_embeddings:
            self.transformer.wte.weight = self.lm_head.weight
        meta = [n for n, t in list(self.named_parameters()) + list(self.named_buffers()) if t.is_meta]
        assert not meta, f"weights left uninitialised: {meta}"
        return self

    def new_kv_cache(self):
        return [KVCache() for _ in self.transformer.h]

    def forward(self, idx, targets=None, ignore_index=-1, kv_cache=None):
        """
        idx:      (B, T) token ids
        targets:  (B, T) next-token ids, positions equal to `ignore_index` are not part of the loss
        kv_cache: list of KVCache (one per layer) from `new_kv_cache()`; idx are the tokens after the cached ones
        returns:  logits (B, T, V) when targets are given, else (B, 1, V) for the last position only
        """
        device = idx.device
        B, T = idx.size()
        start_pos = kv_cache[0].length if kv_cache is not None else 0
        assert start_pos + T <= self.config.block_size, \
            f"Max sequence length is {self.config.block_size}, got {start_pos + T}"

        x = self.transformer.wte(idx)  # (B, T, n_embd)
        if "wpe" in self.transformer:
            pos = torch.arange(start_pos, start_pos + T, dtype=torch.long, device=device)
            x = x + self.transformer.wpe(pos)  # (T, n_embd) broadcast over the batch

        x = self.transformer.drop(x)
        for i, block in enumerate(self.transformer.h):
            x = block(x, kv_cache[i] if kv_cache is not None else None)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=ignore_index)
        else:
            # inference: only the last position is needed
            logits = self.lm_head(x[:, [-1], :])
            loss = None

        return logits, loss

    def crop_block_size(self, block_size):
        """Model surgery: reduce the context length (e.g. load GPT-2 (1024) but use 256)."""
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        if "wpe" in self.transformer:
            self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
            self.transformer.wpe.num_embeddings = block_size
        for module in self.modules():  # modules() visits the shared rotary table once
            if isinstance(module, RotaryEmbedding):
                module.crop(block_size)

    @classmethod
    def from_pretrained(cls, name_or_path, tokenizer=None, override_args=None):
        """
        Load a Hugging Face checkpoint of any supported family (GPT-2, Llama, Qwen2, GPT-NeoX, ...)
        into our from-scratch implementation. See tiny_llm/pretrained.py.
        """
        from tiny_llm.pretrained import load_pretrained
        return load_pretrained(cls, name_or_path, tokenizer, override_args)

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        # 2D params (matmuls, embeddings) are decayed; biases and norms are not
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]

        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)

    @torch.no_grad()
    def stream_generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None,
                        eos_token_id=None, banned_token_ids=None, use_cache=True):
        """
        Yield new token ids one by one, shape (B, 1).
        temperature = 0 -> greedy decoding. Generation stops after `eos_token_id` (int or list of ints).
        `banned_token_ids` can never be sampled (e.g. padding).
        use_cache: keep keys/values of past tokens (much faster). Once the context window is full we fall
        back to re-reading the last `block_size` tokens, exactly like the no-cache version.
        """
        eos_ids = set([eos_token_id] if isinstance(eos_token_id, int) else (eos_token_id or []))
        block_size = self.config.block_size
        cache = None
        for _ in range(max_new_tokens):
            if cache is not None and cache[0].length < block_size:
                logits, _ = self(idx[:, -1:], kv_cache=cache)  # only the newest token
            else:
                cache = self.new_kv_cache() if use_cache else None
                logits, _ = self(idx[:, -block_size:], kv_cache=cache)
            logits = logits[:, -1, :].float()
            if banned_token_ids:
                logits[:, list(banned_token_ids)] = -float('Inf')

            if temperature == 0:
                idx_next = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None and top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('Inf')
                if top_p is not None and 0.0 < top_p < 1.0:
                    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                    cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    remove = cum_probs > top_p
                    remove[:, 1:] = remove[:, :-1].clone()  # always keep the most likely token
                    remove[:, 0] = False
                    logits[remove.scatter(1, sorted_idx, remove)] = -float('Inf')
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)
            yield idx_next
            if eos_ids and idx_next.numel() == 1 and idx_next.item() in eos_ids:
                break

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None, eos_token_id=None,
                 banned_token_ids=None, use_cache=True):
        """Return idx (B, T) extended with up to `max_new_tokens` generated tokens."""
        new_tokens = list(self.stream_generate(idx, max_new_tokens, temperature, top_k, top_p, eos_token_id,
                                               banned_token_ids, use_cache))
        return torch.cat([idx] + new_tokens, dim=1) if new_tokens else idx

In [ ]:
%%writefile tiny_llm/norm.py
import torch
from torch import nn
from torch.nn import functional as F


class LayerNorm(nn.Module):
    """
        Layer Normalization with an optional bias (PyTorch's nn.LayerNorm doesn't support bias=False).
        (different than batch normalization and better in seq problems)   Used by: GPT-2, Pythia.

        y = (x - mean(x)) / sqrt(var(x) + eps) * weight + bias

        Tensor          Type            Shape
        ===========================================================================
        input           float           (batch_size, seq_len, n_embd)
        ---------------------------------------------------------------------------
        output          float           (batch_size, seq_len, n_embd)
        ===========================================================================
    """

    def __init__(self, ndim, bias, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
        self.eps = eps

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, self.eps)


class RMSNorm(nn.Module):
    """
        Root Mean Square normalization: no mean subtraction and no bias, a bit cheaper than LayerNorm
        and works as well.   Used by: Llama, Qwen2, SmolLM2, TinyLlama.

        y = x / sqrt(mean(x^2) + eps) * weight
    """

    def __init__(self, ndim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.eps = eps

    def forward(self, x):
        dtype = x.dtype
        x = x.float()  # normalise in float32 for stability under mixed precision
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * x.to(dtype)


def build_norm(config):
    if config.norm_type == "rmsnorm":
        return RMSNorm(config.n_embd, eps=config.norm_eps)
    return LayerNorm(config.n_embd, bias=config.bias, eps=config.norm_eps)

In [ ]:
%%writefile tiny_llm/pretrained.py
"""Load Hugging Face checkpoints of any supported family (see tiny_llm/families.py) into TinyLLM."""
import math

import torch

from tiny_llm.config import ModelConfig
from tiny_llm.families import get_family
from tiny_llm.tokenizer import SPECIAL_TOKENS

ALLOWED_OVERRIDES = {'drop_rate', 'lora_rank', 'lora_alpha', 'lora_dropout'}


def load_pretrained(cls, name_or_path, tokenizer=None, override_args=None):
    """
    name_or_path: Hugging Face model id ("gpt2", "HuggingFaceTB/SmolLM2-135M", ...) or a local folder.
    tokenizer:    if it contains our special tokens (tiny_llm.tokenizer.create_tokenizer), their embeddings
                  are created: the matrix is extended if needed (padded to a multiple of 8) and the new
                  rows are sampled around the existing embeddings.
    override_args: may change drop_rate and LoRA settings (lora_rank, lora_alpha, lora_dropout).
    """
    from transformers import AutoConfig, AutoModelForCausalLM

    override_args = override_args or {}
    assert set(override_args) <= ALLOWED_OVERRIDES, f"can only override {ALLOWED_OVERRIDES}"

    hf_config = AutoConfig.from_pretrained(name_or_path)
    family = get_family(hf_config.model_type)
    config_args = family.to_config(hf_config)
    config_args.update(override_args)

    model_hf = AutoModelForCausalLM.from_pretrained(name_or_path, dtype=torch.float32)
    base_cfg = ModelConfig(**config_args)
    sd = family.convert(model_hf.state_dict(), base_cfg)
    del model_hf

    # ---- vocabulary: make room for our special tokens
    hf_rows = sd["transformer.wte.weight"].size(0)
    new_ids = _new_token_ids(tokenizer)
    if tokenizer is not None and len(tokenizer) > hf_rows:
        config_args["vocab_size"] = math.ceil(len(tokenizer) / 8) * 8  # multiple of 8: faster matmuls

    tied = config_args.get("tie_embeddings", True)
    with torch.no_grad():
        for name in ("transformer.wte.weight",) if tied else ("transformer.wte.weight", "lm_head.weight"):
            sd[name] = _resize_and_init(sd[name], config_args["vocab_size"], new_ids)
        if tied:
            sd["lm_head.weight"] = sd["transformer.wte.weight"]

    # build on the meta device (no memory, no random init) and plug the converted tensors in directly:
    # peak memory ~ one copy of the weights instead of three
    with torch.device("meta"):
        model = cls(ModelConfig(**config_args))
    missing, unexpected = model.load_state_dict(sd, strict=False, assign=True)
    assert not unexpected, f"unexpected weights: {unexpected}"
    assert all('lora_' in k for k in missing), f"weights missing from the checkpoint: {missing}"
    return model.materialize_()


def _new_token_ids(tokenizer):
    """Ids of our special tokens: they have no trained embedding (new rows or unused spare rows)."""
    if tokenizer is None:
        return []
    added = tokenizer.get_added_vocab()  # unlike convert_tokens_to_ids, never falls back to the unk token
    return sorted(added[t] for t in SPECIAL_TOKENS if t in added)


def _resize_and_init(weight, vocab_size, new_ids):
    """Pad `weight` to `vocab_size` rows; rows of new tokens ~ N(mean, 1e-5 * cov) of the trained rows."""
    if weight.size(0) < vocab_size:
        pad = weight.new_zeros(vocab_size - weight.size(0), weight.size(1))
        weight = torch.cat([weight, pad])
    if new_ids:
        trained = weight[:min(new_ids)].float()  # the original tokenizer's rows
        avg = trained.mean(dim=0)
        centered = trained - avg
        cov = (centered.T @ centered) / (trained.size(0) - 1)
        cov = cov + 1e-6 * torch.eye(cov.size(0))  # keep it positive definite
        dist = torch.distributions.MultivariateNormal(avg, covariance_matrix=1e-5 * cov)
        weight[new_ids] = dist.sample((len(new_ids),)).to(weight.dtype)
    return weight

In [ ]:
%%writefile tiny_llm/rope.py
import torch
from torch import nn


def rotate_half(x):
    """[x1, x2] -> [-x2, x1] on the last dimension."""
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


class RotaryEmbedding(nn.Module):
    """
        Rotary Position Embedding (RoPE, https://arxiv.org/abs/2104.09864).
        Instead of adding a position vector to the input, q and k are rotated by an angle that grows
        with the position, so q·k only depends on the *distance* between two tokens.
        Used by: Llama, Qwen2, SmolLM2, Pythia (partial: only the first `rotary_pct` of each head).

        Tensor          Type            Shape
        ===========================================================================
        q, k            float           (batch_size, n_heads, seq_len, head_dim)
        ===========================================================================
    """

    def __init__(self, head_dim, max_positions, theta=10000.0, rotary_pct=1.0):
        super().__init__()
        self.rot_dim = int(head_dim * rotary_pct)
        assert self.rot_dim % 2 == 0, "the rotated part of a head must have an even size"
        self.theta, self.max_positions = theta, max_positions
        self.reset_buffers()

    def reset_buffers(self, device=None):
        """(Re)compute the cos/sin tables, they are not saved in checkpoints."""
        arange = torch.arange(0, self.rot_dim, 2, dtype=torch.int64, device=device).float()
        inv_freq = 1.0 / (self.theta ** (arange / self.rot_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        t = torch.arange(self.max_positions, dtype=torch.float32, device=device)
        freqs = torch.outer(t, inv_freq)  # (T, rot_dim / 2)
        emb = torch.cat((freqs, freqs), dim=-1)  # (T, rot_dim)
        self.register_buffer("cos", emb.cos(), persistent=False)
        self.register_buffer("sin", emb.sin(), persistent=False)

    def crop(self, max_positions):
        self.max_positions = max_positions
        self.cos, self.sin = self.cos[:max_positions], self.sin[:max_positions]

    def _rotate(self, x, cos, sin):
        x_rot, x_pass = x[..., :self.rot_dim], x[..., self.rot_dim:]
        x_rot = x_rot * cos + rotate_half(x_rot) * sin
        return torch.cat((x_rot, x_pass), dim=-1)

    def forward(self, q, k, start_pos=0):
        T = q.size(-2)
        cos = self.cos[start_pos:start_pos + T].to(q.dtype)
        sin = self.sin[start_pos:start_pos + T].to(q.dtype)
        return self._rotate(q, cos, sin), self._rotate(k, cos, sin)

In [ ]:
%%writefile tiny_llm/runtime.py
"""Device / precision helpers and model initialisation (scratch, resume, or GPT-2 weights)."""
import argparse
import math
import os
from contextlib import nullcontext
from dataclasses import fields

import torch

from tiny_llm.checkpoint import load_checkpoint
from tiny_llm.config import ModelConfig
from tiny_llm.model import TinyLLM
from tiny_llm.tokenizer import create_tokenizer

DTYPES = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}


def resolve_device(device="auto"):
    if device != "auto":
        return device
    if torch.cuda.is_available():
        return 'cuda'
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


def resolve_dtype(dtype, device):
    if dtype != "auto":
        return dtype
    if torch.device(device).type == 'cuda':
        return 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
    return 'float32'


def autocast_context(device, dtype):
    device_type = torch.device(device).type
    if device_type != 'cuda' or dtype == 'float32':
        return nullcontext()
    return torch.amp.autocast(device_type=device_type, dtype=DTYPES[dtype])


def setup_torch(seed):
    torch.manual_seed(seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def init_model(init_from, model_config, tokenizer_name="", ckpt_path=None, device='cpu'):
    """
    Build the model and its tokenizer.
        init_from = "scratch"  -> random weights with `model_config` (tokenizer: `tokenizer_name` or gpt2)
        init_from = "resume"   -> weights + config + tokenizer name from `ckpt_path`
        otherwise              -> a Hugging Face model id / folder of a supported family (tiny_llm/families.py),
                                  its vocabulary extended with our special tokens
    Returns (model, tokenizer, checkpoint or None).
    """
    checkpoint = None
    if init_from == 'scratch':
        tokenizer = create_tokenizer(tokenizer_name or "gpt2")
        if model_config.vocab_size < len(tokenizer):
            model_config.vocab_size = math.ceil(len(tokenizer) / 64) * 64
            print(f"vocab_size raised to {model_config.vocab_size} to fit the tokenizer")
        model = TinyLLM(model_config)
    elif init_from == 'resume':
        assert ckpt_path and os.path.exists(ckpt_path), f"no checkpoint found at {ckpt_path}"
        model, checkpoint = load_checkpoint(ckpt_path, device)
        print(f"checkpoint loaded from {ckpt_path} :)")
        tokenizer = create_tokenizer(tokenizer_name or checkpoint.get('tokenizer', 'gpt2'))
        model = _apply_runtime_overrides(model, model_config)
    else:
        tokenizer = create_tokenizer(tokenizer_name or init_from)
        override = dict(drop_rate=model_config.drop_rate, lora_rank=model_config.lora_rank,
                        lora_alpha=model_config.lora_alpha, lora_dropout=model_config.lora_dropout)
        model = TinyLLM.from_pretrained(init_from, tokenizer, override)
        print(f"pre-trained weights loaded from {init_from} :)")

    assert len(tokenizer) <= model.config.vocab_size, \
        f"the tokenizer ({len(tokenizer)} tokens) doesn't fit in the model vocabulary ({model.config.vocab_size})"
    # crop the context length of a pre-trained model if we asked for a smaller one
    if model_config.block_size < model.config.block_size:
        model.crop_block_size(model_config.block_size)
    return model.to(device), tokenizer, checkpoint


def _apply_runtime_overrides(model, model_config):
    """
    Dropout and (newly added) LoRA layers are not part of the learned weights, so a resumed model takes
    them from the requested config. This is what lets us LoRA fine-tune one of our own checkpoints.
    """
    new_args = model.config.to_dict()
    new_args['drop_rate'] = model_config.drop_rate
    if model.config.lora_rank == 0 and model_config.lora_rank > 0:
        new_args.update(lora_rank=model_config.lora_rank, lora_alpha=model_config.lora_alpha,
                        lora_dropout=model_config.lora_dropout)
    if new_args == model.config.to_dict():
        return model

    device = next(model.parameters()).device
    new_model = TinyLLM(ModelConfig.from_dict(new_args)).to(device)
    missing, unexpected = new_model.load_state_dict(model.state_dict(), strict=False)
    assert not unexpected and all('lora_' in k for k in missing), (missing, unexpected)
    return new_model


def parse_config_args(*dataclasses, description=None, argv=None):
    """
    Expose every field of the given dataclasses as a `--name value` CLI flag.
    Returns one instance per dataclass, e.g. `train_cfg, model_cfg = parse_config_args(TrainConfig, ModelConfig)`.
    """
    parser = argparse.ArgumentParser(description=description,
                                     formatter_class=argparse.ArgumentDefaultsHelpFormatter)

    def str2bool(v):
        if v.lower() in ('1', 'true', 'yes', 'y'):
            return True
        if v.lower() in ('0', 'false', 'no', 'n'):
            return False
        raise argparse.ArgumentTypeError(f"expected a boolean, got {v!r}")

    defaults = [dc() for dc in dataclasses]
    seen = set()
    for default in defaults:
        group = parser.add_argument_group(type(default).__name__)
        for f in fields(default):
            if f.name in seen:
                continue
            seen.add(f.name)
            value = getattr(default, f.name)
            arg_type = str2bool if isinstance(value, bool) else type(value)
            group.add_argument(f"--{f.name}", type=arg_type, default=value)
    args = vars(parser.parse_args(argv))
    return [type(d)(**{f.name: args[f.name] for f in fields(d)}) for d in defaults]

In [ ]:
%%writefile tiny_llm/tokenizer.py
"""
Any Hugging Face tokenizer + the special tokens used to format question/answer samples.

The tokenizer's own bos/eos tokens are kept (Llama tokenizers prepend <s>, generation still stops at the
model's native end of text), our tokens are added on top.
"""
from transformers import AutoTokenizer

PAD, BOS, EOS, QUES, ANS = "<pad>", "<sost>", "<eost>", "<ques>", "<ans>"
SPECIAL_TOKENS = [PAD, BOS, EOS, QUES, ANS]

# end-of-text tokens used by the supported families (base and chat models)
NATIVE_STOP_TOKENS = ["<|endoftext|>", "</s>", "<|im_end|>", "<|end_of_text|>", "<|eot_id|>"]


def create_tokenizer(name="gpt2"):
    tokenizer = AutoTokenizer.from_pretrained(name)
    # for GPT-2 this keeps the ids of the first version: <pad>=50257 <sost>=50258 <eost>=50259 <ques>=50260 <ans>=50261
    tokenizer.add_special_tokens({'pad_token': PAD, 'additional_special_tokens': [BOS, EOS, QUES, ANS]})
    # we handle truncation ourselves, silence the "sequence too long" warning
    tokenizer.model_max_length = int(1e9)
    return tokenizer


def stop_token_ids(tokenizer):
    """<eost> for our fine-tuned models + the native end-of-text tokens of the base model."""
    vocab = tokenizer.get_vocab()
    ids = {vocab[t] for t in [EOS] + NATIVE_STOP_TOKENS if t in vocab}
    if tokenizer.eos_token_id is not None:
        ids.add(tokenizer.eos_token_id)
    return sorted(ids)


def format_qa_prompt(question, context=""):
    """The exact text the model sees before it starts writing the answer."""
    return f"{BOS}{context.strip()}{QUES}{question.strip()}{ANS}"


def format_qa_sample(question, answer, context=""):
    """A full training sample: prompt + answer + end of text."""
    return f"{format_qa_prompt(question, context)}{answer.strip()}{EOS}"

In [ ]:
%%writefile tiny_llm/training.py
"""Training utilities shared by pre-training (scripts/train.py) and fine-tuning (scripts/finetune.py)."""
import math
import os
import time

import torch

from tiny_llm.checkpoint import save_checkpoint


def get_lr(it, cfg):
    """Linear warmup, then cosine decay down to min_lr."""
    if it < cfg.warmup_iters:
        return cfg.learning_rate * (it + 1) / (cfg.warmup_iters + 1)
    if it > cfg.lr_decay_iters:
        return cfg.min_lr
    decay_ratio = (it - cfg.warmup_iters) / max(1, cfg.lr_decay_iters - cfg.warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))  # goes from 1 to 0
    return cfg.min_lr + coeff * (cfg.learning_rate - cfg.min_lr)


@torch.no_grad()
def estimate_loss(model, get_batch_fns, eval_iters, ctx):
    """Average loss over `eval_iters` batches for each split. get_batch_fns = {"train": fn, "val": fn}."""
    out = {}
    model.eval()
    for split, get_batch in get_batch_fns.items():
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch()
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train(model, optimizer, cfg, get_batch_fns, device, ctx, dtype, iter_num=0, best_val_loss=float('inf'),
          ckpt_extra=None):
    """
    The optimisation loop: gradient accumulation, mixed precision, grad clipping, lr schedule,
    periodic evaluation and checkpointing (best val loss -> `{out_dir}/{ckpt_name}`).
    Returns the best validation loss.
    """
    ckpt_path = os.path.join(cfg.out_dir, cfg.ckpt_name)
    raw_model = model
    if cfg.compile:
        print("compiling the model... (takes a ~minute)")
        model = torch.compile(model)

    # a GradScaler is only needed for float16 (bfloat16 has the range of float32)
    scaler = torch.amp.GradScaler(torch.device(device).type, enabled=(dtype == 'float16'))
    trainable = [p for p in model.parameters() if p.requires_grad]

    X, Y = get_batch_fns['train']()
    t0 = time.time()
    while True:
        lr = get_lr(iter_num, cfg) if cfg.decay_lr else cfg.learning_rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # evaluate the loss on train/val sets and write checkpoints
        if iter_num % cfg.eval_interval == 0:
            losses = estimate_loss(model, get_batch_fns, cfg.eval_iters, ctx)
            print(f"Val-Step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
            if losses['val'] < best_val_loss or cfg.always_save_checkpoint:
                best_val_loss = min(best_val_loss, losses['val'])
                if iter_num > 0:
                    save_checkpoint(ckpt_path, raw_model, optimizer, raw_model.config.to_dict(),
                                    iter_num, best_val_loss, extra=ckpt_extra)

        if iter_num >= cfg.max_iters:
            break

        # forward / backward with gradient accumulation (simulates a bigger batch)
        for _ in range(cfg.grad_accum_steps):
            with ctx:
                _, loss = model(X, Y)
                loss = loss / cfg.grad_accum_steps
            X, Y = get_batch_fns['train']()  # prefetch next batch while the GPU works
            scaler.scale(loss).backward()

        if cfg.grad_clip != 0.0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, cfg.grad_clip)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        t1 = time.time()
        dt = t1 - t0
        t0 = t1
        if iter_num % cfg.log_interval == 0:
            lossf = loss.item() * cfg.grad_accum_steps
            print(f"iter {iter_num}: loss {lossf:.4f}, lr {lr:.2e}, time {dt * 1000:.2f}ms")
        iter_num += 1

    return best_val_loss


def build_optimizer(model, cfg, device):
    return model.configure_optimizers(cfg.weight_decay, cfg.learning_rate, (cfg.beta1, cfg.beta2),
                                      torch.device(device).type)

In [ ]:
%%writefile scripts/ask.py
"""
Ask a question to a fine-tuned checkpoint (see scripts/finetune.py).

    python scripts/ask.py --ckpt ./checkpoints/finetuned.pt --question "Who wrote the paper?" \
        --context "The paper was written by Ismail Ben Alla in 2023."
"""
import argparse

import torch

from tiny_llm import answer_me
from tiny_llm.runtime import resolve_device
from generate import load_model


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--ckpt", required=True)
    parser.add_argument("--question", required=True)
    parser.add_argument("--context", default="")
    parser.add_argument("--max_new_tokens", type=int, default=100)
    parser.add_argument("--temperature", type=float, default=0.7)
    parser.add_argument("--top_k", type=int, default=50)
    parser.add_argument("--top_p", type=float, default=0.95)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--seed", type=int, default=2000)
    args = parser.parse_args(argv)

    torch.manual_seed(args.seed)
    device = resolve_device(args.device)
    model, tokenizer = load_model(args.ckpt, device=device)
    print(answer_me(model, tokenizer, args.question, args.context, max_new_tokens=args.max_new_tokens,
                    temperature=args.temperature, top_k=args.top_k, top_p=args.top_p))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/eval_toy.py
"""
Exact-match accuracy on the toy datasets (see make_toy_data.py), with greedy decoding.

    python scripts/eval_toy.py arithmetic --ckpt ./checkpoints/arithmetic.pt --data ./data/toy/arithmetic/test.json
    python scripts/eval_toy.py qa --ckpt ./checkpoints/finetuned.pt --data ./data/toy/qa/val.json
    python scripts/eval_toy.py qa --pretrained gpt2 --data ./data/toy/qa/val.json      # baseline, no fine-tuning
"""
import argparse
import json

import torch

from tiny_llm import answer_me, stream_tokens
from tiny_llm.runtime import resolve_device
from generate import load_model


def evaluate(task, model, tokenizer, samples, verbose=3):
    correct = 0
    for i, s in enumerate(samples):
        if task == "arithmetic":
            out = "".join(stream_tokens(model, tokenizer, s["prompt"], max_new_tokens=6, temperature=0))
            pred = out.split("\n")[0].strip()
        else:
            pred = answer_me(model, tokenizer, s["question"], s["context"], max_new_tokens=12, temperature=0)
        ok = pred == s["answer"]
        correct += ok
        if i < verbose:
            question = s.get("prompt") or s["question"]
            print(f"  {question!r} -> {pred!r} (expected {s['answer']!r}) {'✅' if ok else '❌'}")
    return correct / len(samples)


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("task", choices=["arithmetic", "qa"])
    source = parser.add_mutually_exclusive_group(required=True)
    source.add_argument("--ckpt")
    source.add_argument("--pretrained")
    parser.add_argument("--data", required=True)
    parser.add_argument("--limit", type=int, default=0, help="evaluate only the first N samples")
    parser.add_argument("--device", default="auto")
    args = parser.parse_args(argv)

    torch.manual_seed(0)
    model, tokenizer = load_model(args.ckpt, args.pretrained, resolve_device(args.device))
    with open(args.data) as f:
        samples = json.load(f)
    samples = samples[:args.limit] if args.limit else samples
    acc = evaluate(args.task, model, tokenizer, samples)
    print(f"{args.task} exact-match accuracy: {acc:.1%} on {len(samples)} samples")
    return acc


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/finetune.py
"""
Fine-tune TinyLLM for (context-)question answering, optionally with LoRA.
Data: data_dir/train.json and data_dir/val.json, lists of {"context", "question", "answer"}.

    # LoRA fine-tuning of GPT-2 small (it is better to fine tune a pre-trained LLM)
    python scripts/finetune.py --init_from gpt2 --data_dir ./data/qa --lora_rank 8 \
        --learning_rate 3e-4 --drop_rate 0.1 --max_iters 2000 --warmup_iters 100 --lr_decay_iters 2000

    # LoRA fine-tuning of other small LLMs (Llama / Qwen2 / Pythia families)
    python scripts/finetune.py --init_from HuggingFaceTB/SmolLM2-135M --data_dir ./data/qa --lora_rank 8 ...

    # full fine-tuning of one of our own pre-trained checkpoints
    python scripts/finetune.py --init_from resume --resume_from ./checkpoints/ckpt.pt --learning_rate 3e-5
"""
import os
import sys

from tiny_llm.config import ModelConfig, TrainConfig
from tiny_llm.data import BatchLoader
from tiny_llm.lora import mark_only_lora_as_trainable
from tiny_llm.runtime import autocast_context, init_model, parse_config_args, resolve_device, resolve_dtype, setup_torch
from tiny_llm.training import build_optimizer, train

FINETUNE_DEFAULTS = dict(init_from="gpt2", learning_rate=3e-5, warmup_iters=100, lr_decay_iters=5000,
                         max_iters=5000, min_lr=3e-6, grad_accum_steps=4, eval_interval=100, eval_iters=50,
                         log_interval=10, ckpt_name="finetuned.pt")


def main(argv=None):
    # fine-tuning has its own defaults, still overridable from the command line
    cfg, model_cfg = parse_config_args(TrainConfig, ModelConfig, description=__doc__,
                                       argv=_with_defaults(argv, FINETUNE_DEFAULTS))
    setup_torch(cfg.seed)
    device = resolve_device(cfg.device)
    dtype = resolve_dtype(cfg.dtype, device)
    ctx = autocast_context(device, dtype)

    ckpt_in = cfg.resume_from or os.path.join(cfg.out_dir, cfg.ckpt_name)
    model, tokenizer, checkpoint = init_model(cfg.init_from, model_cfg, cfg.tokenizer, ckpt_in, device)

    if model.config.lora_rank > 0:
        # LoRA: freeze the LLM, train only A and B (+ the tied embeddings so new special tokens are learned)
        mark_only_lora_as_trainable(model, extra_trainable=("wte",) if cfg.train_embeddings else ())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {n_train / 1e6:.2f}M / {n_total / 1e6:.2f}M | device: {device} | dtype: {dtype}")

    # create dataloader for both train and val datasets
    loader_args = dict(tokenizer=tokenizer, toks_in_batch=cfg.toks_per_batch, device=device,
                       max_len=min(cfg.max_seq_len, model.config.block_size + 1),
                       answer_only=cfg.loss_on_answer_only, seed=cfg.seed)
    train_loader = BatchLoader(cfg.data_dir, "train", **loader_args)
    val_loader = BatchLoader(cfg.data_dir, "val", **loader_args)
    print(f"{len(train_loader)} train batches, {len(val_loader)} val batches")

    optimizer = build_optimizer(model, cfg, device)
    iter_num, best_val_loss = 0, float('inf')
    if checkpoint is not None and checkpoint.get('stage') == 'finetune':
        # resuming an interrupted fine-tuning run
        optimizer.load_state_dict(checkpoint['optimizer'])
        iter_num, best_val_loss = checkpoint['iter_num'], checkpoint['best_val_loss']
    checkpoint = None

    get_batch_fns = {"train": lambda: next(train_loader), "val": lambda: next(val_loader)}
    best = train(model, optimizer, cfg, get_batch_fns, device, ctx, dtype, iter_num, best_val_loss,
                 ckpt_extra={'stage': 'finetune', 'tokenizer': tokenizer.name_or_path})
    print(f"done, best val loss {best:.4f}")


def _with_defaults(argv, defaults):
    """Prepend `--key value` defaults so that flags given by the user (which come later) win."""
    argv = list(sys.argv[1:] if argv is None else argv)
    prefix = []
    for key, value in defaults.items():
        prefix += [f"--{key}", str(value)]
    return prefix + argv


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/generate.py
"""
Text completion with a trained checkpoint or with the original GPT-2 weights.

    python scripts/generate.py --ckpt ./checkpoints/ckpt.pt --prompt "ismail ben alla is the next elon mask because"
    python scripts/generate.py --pretrained gpt2 --prompt "Once upon a time"
    python scripts/generate.py --pretrained HuggingFaceTB/SmolLM2-135M --prompt "Once upon a time"
"""
import argparse

import torch

from tiny_llm import TinyLLM, create_tokenizer, generate_text, load_checkpoint, merge_lora
from tiny_llm.runtime import resolve_device


def load_model(ckpt=None, pretrained=None, device="cpu"):
    """A checkpoint of ours (its tokenizer name is stored inside) or any supported Hugging Face model."""
    if ckpt:
        model, checkpoint = load_checkpoint(ckpt, device)
        tokenizer = create_tokenizer(checkpoint.get("tokenizer", "gpt2"))
    else:
        tokenizer = create_tokenizer(pretrained or "gpt2")
        model = TinyLLM.from_pretrained(pretrained or "gpt2", tokenizer).to(device)
    return merge_lora(model).eval(), tokenizer


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    source = parser.add_mutually_exclusive_group()
    source.add_argument("--ckpt", help="path to a checkpoint written by train.py / finetune.py")
    source.add_argument("--pretrained", default="gpt2", help="Hugging Face id of a supported model (see tiny_llm/families.py)")
    parser.add_argument("--prompt", default="ismail ben alla is the next elon mask because")
    parser.add_argument("--num_samples", type=int, default=1)
    parser.add_argument("--max_new_tokens", type=int, default=50)
    parser.add_argument("--temperature", type=float, default=1.0)
    parser.add_argument("--top_k", type=int, default=100)
    parser.add_argument("--top_p", type=float, default=None)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--seed", type=int, default=2000)
    args = parser.parse_args(argv)

    torch.manual_seed(args.seed)
    device = resolve_device(args.device)
    model, tokenizer = load_model(args.ckpt, args.pretrained, device)

    # generate k samples
    for k in range(args.num_samples):
        print(f"sample num:[{k}]")
        print(generate_text(model, tokenizer, args.prompt, args.max_new_tokens, args.temperature, args.top_k,
                            args.top_p, device))
        print("-" * 60)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/list_families.py
"""Print the supported model families and what changes for each of them compared to GPT-2."""
from tiny_llm.families import markdown_table

if __name__ == "__main__":
    print(markdown_table())

In [ ]:
%%writefile scripts/make_toy_data.py
"""
Small synthetic datasets to check, in minutes on a CPU, that the model really learns.

    arithmetic/  train.bin, val.bin      "a+b=c" lines (pre-training from scratch), 10% of the (a, b) pairs
                 test.json               are never seen during training and are used to measure accuracy
    qa/          train.json, val.json    context / question / answer about made-up people; val uses
                                         names, cities and jobs never seen in train (needs real reading, not memory)

    python scripts/make_toy_data.py --out ./data/toy
"""
import argparse
import json
import os
import random

import numpy as np
import tiktoken

FIRST = ["Amina", "Youssef", "Sara", "Omar", "Lina", "Karim", "Nora", "Adam", "Hiba", "Ilyas", "Maya", "Rayan",
         "Salma", "Hamza", "Ines", "Zakaria", "Yasmine", "Anas", "Kenza", "Mehdi", "Aya", "Bilal", "Rania", "Taha",
         "Leila", "Samir", "Dounia", "Imane", "Walid", "Farah", "Nabil", "Siham"]
CITIES = ["Rabat", "Paris", "Tokyo", "Cairo", "Lima", "Oslo", "Madrid", "Dakar", "Lagos", "Rome", "Berlin", "Quito",
          "Seoul", "Lisbon", "Nairobi", "Vienna", "Dublin", "Havana", "Prague", "Tunis", "Athens", "Bogota"]
JOBS = ["doctor", "teacher", "pilot", "baker", "farmer", "nurse", "painter", "lawyer", "chef", "engineer",
        "dentist", "writer", "singer", "plumber", "architect", "librarian", "carpenter", "journalist"]
COLORS = ["red", "blue", "green", "yellow", "purple", "orange", "black", "white", "pink", "brown"]

TEMPLATE = "{name} is a {job} who lives in {city}. Their favourite color is {color}."
QUESTIONS = [
    ("Where does {name} live?", "{city}"),
    ("What is the job of {name}?", "{job}"),
    ("What is the favourite color of {name}?", "{color}"),
    ("Who lives in {city}?", "{name}"),
]


def make_arithmetic(out, rng, enc, max_n=100, test_ratio=0.1):
    pairs = [(a, b) for a in range(max_n) for b in range(max_n)]
    rng.shuffle(pairs)
    n_test = int(len(pairs) * test_ratio)
    test, train = pairs[:n_test], pairs[n_test:]

    def write(name, items, repeats):
        lines = [f"{a}+{b}={a + b}\n" for _ in range(repeats) for a, b in rng.sample(items, len(items))]
        ids = enc.encode_ordinary("".join(lines))
        np.array(ids, dtype=np.uint16).tofile(os.path.join(out, f"{name}.bin"))
        return len(ids)

    os.makedirs(out, exist_ok=True)
    n_train = write("train", train, repeats=3)
    n_val = write("val", test, repeats=1)  # loss on unseen pairs
    with open(os.path.join(out, "test.json"), "w") as f:
        json.dump([{"prompt": f"{a}+{b}=", "answer": str(a + b)} for a, b in test], f)
    print(f"✅ arithmetic: {n_train:,} train tokens, {n_val:,} val tokens, {len(test)} unseen test problems")


def make_qa(out, rng, n_train=600, n_val=100):
    def split(items, k):
        items = items[:]
        rng.shuffle(items)
        return items[k:], items[:k]

    # val people / cities / jobs never appear in train
    names_tr, names_va = split(FIRST, 8)
    cities_tr, cities_va = split(CITIES, 6)
    jobs_tr, jobs_va = split(JOBS, 5)

    def sample(names, cities, jobs):
        name, city, job, color = rng.choice(names), rng.choice(cities), rng.choice(jobs), rng.choice(COLORS)
        fields = dict(name=name, city=city, job=job, color=color)
        # a distractor sentence about someone else makes the model actually read the question
        other = dict(name=rng.choice([n for n in names if n != name]), city=rng.choice([c for c in cities if c != city]),
                     job=rng.choice(jobs), color=rng.choice(COLORS))
        sentences = [TEMPLATE.format(**fields), TEMPLATE.format(**other)]
        rng.shuffle(sentences)
        q, a = rng.choice(QUESTIONS)
        return {"context": " ".join(sentences), "question": q.format(**fields), "answer": a.format(**fields)}

    os.makedirs(out, exist_ok=True)
    for name, n, pools in (("train", n_train, (names_tr, cities_tr, jobs_tr)),
                           ("val", n_val, (names_va, cities_va, jobs_va))):
        with open(os.path.join(out, f"{name}.json"), "w") as f:
            json.dump([sample(*pools) for _ in range(n)], f, indent=1)
    print(f"✅ qa: {n_train} train / {n_val} val samples (val names, cities and jobs are unseen)")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--out", default="./data/toy")
    parser.add_argument("--seed", type=int, default=2000)
    args = parser.parse_args(argv)
    rng = random.Random(args.seed)
    make_arithmetic(os.path.join(args.out, "arithmetic"), rng, tiktoken.get_encoding("r50k_base"))
    make_qa(os.path.join(args.out, "qa"), rng)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/prepare_data.py
"""
Turn a folder of .txt files (e.g. OpenWebText) into train.bin / val.bin / test.bin token streams.
Each document is GPT-2 tokenized (r50k_base) and followed by <|endoftext|>; ids are stored as uint16.

    python scripts/prepare_data.py --src ./raw_txt --dst ./data/openwebtext --val_ratio 0.1 --test_ratio 0.1
"""
import argparse
import os
import random

import numpy as np
import tiktoken


def write_split(docs, enc, path):
    eot = enc.eot_token
    # first pass: count tokens, second pass: write them into a memmap (keeps memory flat)
    lengths = [len(ids) + 1 for ids in enc.encode_ordinary_batch(docs)]
    total = int(np.sum(lengths, dtype=np.uint64))
    arr = np.memmap(path, dtype=np.uint16, mode='w+', shape=(total,))
    pos = 0
    for ids in enc.encode_ordinary_batch(docs):
        ids.append(eot)
        arr[pos:pos + len(ids)] = ids
        pos += len(ids)
    arr.flush()
    print(f"✅ {path}: {len(docs)} documents, {total:,} tokens")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--src", required=True, help="folder with .txt files (searched recursively)")
    parser.add_argument("--dst", required=True)
    parser.add_argument("--val_ratio", type=float, default=0.1)
    parser.add_argument("--test_ratio", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=2000)
    args = parser.parse_args(argv)

    files = sorted(os.path.join(root, f) for root, _, names in os.walk(args.src) for f in names if f.endswith(".txt"))
    assert files, f"no .txt files found in {args.src}"
    random.Random(args.seed).shuffle(files)

    n_val = max(1, int(len(files) * args.val_ratio))
    n_test = int(len(files) * args.test_ratio)
    splits = {"val": files[:n_val], "test": files[n_val:n_val + n_test], "train": files[n_val + n_test:]}
    assert splits["train"], "not enough files for a train split"

    enc = tiktoken.get_encoding("r50k_base")
    os.makedirs(args.dst, exist_ok=True)
    for split, paths in splits.items():
        if not paths:
            continue
        docs = []
        for p in paths:
            with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                docs.append(f.read())
        write_split(docs, enc, os.path.join(args.dst, f"{split}.bin"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/train.py
"""
Pre-train TinyLLM on a flat token stream (data_dir/train.bin, data_dir/val.bin).

    python scripts/train.py --data_dir ./data/openwebtext --init_from scratch
    python scripts/train.py --init_from resume                  # continue from out_dir/ckpt_name
    python scripts/train.py --init_from HuggingFaceTB/SmolLM2-135M   # continue pre-training another small LLM
    python scripts/train.py --n_layer 4 --n_head 4 --n_embd 256 --block_size 256 --batch_size 8   # small run
"""
import os

from tiny_llm.config import ModelConfig, TrainConfig
from tiny_llm.data import get_batch, load_bin
from tiny_llm.runtime import autocast_context, init_model, parse_config_args, resolve_device, resolve_dtype, setup_torch
from tiny_llm.training import build_optimizer, train


def main(argv=None):
    cfg, model_cfg = parse_config_args(TrainConfig, ModelConfig, description=__doc__, argv=argv)
    setup_torch(cfg.seed)
    device = resolve_device(cfg.device)
    dtype = resolve_dtype(cfg.dtype, device)
    ctx = autocast_context(device, dtype)

    # connect to dataset
    train_data = load_bin(cfg.data_dir, "train")
    val_data = load_bin(cfg.data_dir, "val")

    ckpt_path = cfg.resume_from or os.path.join(cfg.out_dir, cfg.ckpt_name)
    model, tokenizer, checkpoint = init_model(cfg.init_from, model_cfg, cfg.tokenizer, ckpt_path, device)
    block_size = model.config.block_size
    print(f"number of parameters: {model.get_num_params() / 1e6:.2f}M | device: {device} | dtype: {dtype}")

    optimizer = build_optimizer(model, cfg, device)
    iter_num, best_val_loss = 0, float('inf')
    if checkpoint is not None:
        if checkpoint.get('optimizer'):
            optimizer.load_state_dict(checkpoint['optimizer'])
        iter_num, best_val_loss = checkpoint['iter_num'], checkpoint['best_val_loss']
        checkpoint = None  # free memory

    get_batch_fns = {
        "train": lambda: get_batch(train_data, cfg.batch_size, block_size, device),
        "val": lambda: get_batch(val_data, cfg.batch_size, block_size, device),
    }
    best = train(model, optimizer, cfg, get_batch_fns, device, ctx, dtype, iter_num, best_val_loss,
                 ckpt_extra={'stage': 'pretrain', 'tokenizer': tokenizer.name_or_path})
    print(f"done, best val loss {best:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
sys.path.insert(0, WORK)
import tiny_llm
from tiny_llm.families import markdown_table
from IPython.display import Markdown, display

print("tiny_llm", tiny_llm.__version__)
display(Markdown(markdown_table()))  # what changes for each supported model family

## 3. Sanity check on simple data

Two small synthetic tasks that prove the model learns:
**addition** learned from scratch (scored on problems never seen in training) and
**reading comprehension** with LoRA fine-tuning (scored on people, cities and jobs never seen in training).

In [ ]:
!python scripts/make_toy_data.py --out data/toy

In [ ]:
!python scripts/train.py --data_dir data/toy/arithmetic --ckpt_name arithmetic.pt \
    --n_layer 4 --n_head 4 --n_embd 128 --block_size 64 --batch_size 64 --grad_accum_steps 1 \
    --learning_rate 2e-3 --min_lr 2e-4 --warmup_iters 100 --lr_decay_iters 3000 --max_iters 3000 \
    --eval_interval 500 --eval_iters 20 --log_interval 100
!python scripts/eval_toy.py arithmetic --ckpt checkpoints/arithmetic.pt --data data/toy/arithmetic/test.json

In [ ]:
!python scripts/eval_toy.py qa --pretrained distilgpt2 --data data/toy/qa/val.json   # before fine-tuning
!python scripts/finetune.py --init_from distilgpt2 --data_dir data/toy/qa --lora_rank 8 --learning_rate 1e-3 \
    --max_iters 300 --warmup_iters 20 --lr_decay_iters 300 --grad_accum_steps 1 --toks_per_batch 2000 \
    --eval_interval 100 --eval_iters 10 --ckpt_name qa_finetuned.pt
!python scripts/eval_toy.py qa --ckpt checkpoints/qa_finetuned.pt --data data/toy/qa/val.json

## 4. Pre-training on OpenWebText

Add the Kaggle dataset **benallaismail/gpt-data** (a quarter of OpenWebText, already tokenized into
`train.bin` / `val.bin`) to the notebook with *Add Input*. Without it this cell is skipped.
Increase `--max_iters` for a real run, `--init_from resume` continues from the last checkpoint.

In [ ]:
import glob
bins = glob.glob("/kaggle/input/**/train.bin", recursive=True)
if bins:
    DATA_DIR = os.path.dirname(bins[0])
    print("data:", DATA_DIR)
    !python scripts/train.py --data_dir {DATA_DIR} --ckpt_name owt.pt \
        --n_layer 6 --n_head 6 --n_embd 384 --block_size 256 --batch_size 32 --grad_accum_steps 4 \
        --learning_rate 1e-3 --min_lr 1e-4 --warmup_iters 200 --lr_decay_iters 5000 --max_iters 5000 \
        --eval_interval 500 --eval_iters 50 --log_interval 50
    !python scripts/generate.py --ckpt checkpoints/owt.pt --prompt "ismail ben alla is the next elon mask because"
else:
    print("dataset not attached, skipping (Add Input -> benallaismail/gpt-data)")

## 5. Chat with another small LLM

Tiny LLM is not limited to GPT-2: any supported family (Llama, Qwen2, GPT-NeoX) loads into the same
from-scratch code. Here SmolLM2-135M-Instruct (Llama family) answers through its own chat template.

In [ ]:
from tiny_llm import TinyLLM, create_tokenizer, stream_chat

name = "HuggingFaceTB/SmolLM2-135M-Instruct"   # or "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = create_tokenizer(name)
model = TinyLLM.from_pretrained(name, tokenizer).to(device).eval()

messages = [{"role": "user", "content": "Explain in two sentences what a transformer is."}]
for piece in stream_chat(model, tokenizer, messages, max_new_tokens=80, temperature=0.3):
    print(piece, end="", flush=True)

## 6. Download the checkpoints

In [ ]:
import shutil
if os.path.isdir("checkpoints"):
    archive = shutil.make_archive("tiny-llm-checkpoints", "zip", "checkpoints")
    print("created", archive)
    if ON_COLAB:
        from google.colab import files
        files.download(archive)
    elif ON_KAGGLE:
        print("Kaggle: click 'Save Version', then download it from the notebook's Output tab.")